# Qwen2.5 Opposing Counsel Training Notebook

Upgraded training pipeline with:
- Assistant-only loss masking
- Packing enabled
- Better LoRA target modules
- EOS handling
- Improved SFTConfig setup
- Resume-safe training

In [1]:

# Install CUDA-enabled PyTorch first (Windows, CUDA 12.1 wheels)
!pip uninstall -y torch torchvision torchaudio
!pip install -q --index-url https://download.pytorch.org/whl/cu128 torch torchvision torchaudio

# Then install training stack
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece


Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import inspect
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("Torch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


c:\Users\Prakhar Parashar\Documents\Opposing_counsel_model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.11.0+cu128
Torch CUDA build: 12.8
CUDA Available: True
GPU count: 1
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [3]:

# =========================
# CONFIG
# =========================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

DATASET_PATH = "property_litigation_opposing_counsel_dataset_3000_updated.jsonl"

OUTPUT_DIR = "./qwen-opposing-counsel-v1-r32-512-packtrue"

MAX_SEQ_LENGTH = 512

BATCH_SIZE = 1
GRAD_ACCUM = 8

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

print("Config loaded")


Config loaded


In [4]:

# =========================
# LOAD DATASET
# =========================

dataset = load_dataset(
    "json",
    data_files=DATASET_PATH,
    split="train"
)

print(dataset)

# Train / test split
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(dataset)


Generating train split: 2995 examples [00:00, 187256.88 examples/s]

Dataset({
    features: ['messages'],
    num_rows: 2995
})
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2845
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 150
    })
})


In [5]:

# =========================
# LOAD TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")


Tokenizer loaded


In [6]:

# =========================
# DATASET FORMAT NOTE
# =========================

# No formatting_func is used.
# The dataset already has a `messages` column, so TRL applies the chat template.


In [7]:

# =========================
# 4-BIT QUANTIZATION
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded")


W0513 13:16:49.639000 22488 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/434 [00:00<04:46,  1.51it/s]c:\Users\Prakhar Parashar\Documents\Opposing_counsel_model\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:07<00:00, 61.46it/s]


Model loaded


In [8]:

# =========================
# LoRA CONFIG
# =========================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()


trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


In [12]:

# =========================
# TRAINER CONFIG
# =========================

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=5,
    eval_steps=20,
    save_steps=20,
    save_total_limit=2,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    # max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    assistant_only_loss=True,
    eos_token="<|im_end|>",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    report_to="none",
)

sft_signature = inspect.signature(SFTConfig.__init__).parameters

if "eval_strategy" in sft_signature:
    sft_kwargs["eval_strategy"] = "steps"
else:
    sft_kwargs["evaluation_strategy"] = "steps"

if "gradient_checkpointing_kwargs" in sft_signature:
    sft_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

sft_config = SFTConfig(**sft_kwargs)

print("SFT config ready")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFT config ready


In [13]:

# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    args=sft_config,
)

print("Trainer ready")


[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Trainer ready


In [14]:

# =========================
# AUTO RESUME TRAINING
# =========================

from transformers.trainer_utils import get_last_checkpoint

checkpoint = None

if os.path.isdir(OUTPUT_DIR):
    checkpoint = get_last_checkpoint(OUTPUT_DIR)

if checkpoint:
    print(f"Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print("Starting fresh training")
    trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting fresh training


Step,Training Loss,Validation Loss
20,1.667199,1.663894
40,1.563237,1.547708
60,1.473769,1.486987
80,1.321089,1.450498
100,1.176079,1.448081
120,1.168417,1.435551
140,1.157859,1.405732
160,0.977004,1.425501
180,0.981714,1.458829
200,0.953792,1.446477


In [15]:

# =========================
# SAVE MODEL
# =========================

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved")


Model saved


In [16]:

# =========================
# INFERENCE TEST
# =========================

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

messages = [
    {
        "role": "system",
        "content": "You are adversarial Indian opposing counsel in a property-litigation training simulation. Critically challenge the user's argument using title analysis, possession analysis, civil procedure, evidentiary scrutiny, burden-of-proof evaluation, contradiction exposure, and fact-specific legal reasoning. Do not invent case citations."
    },
    {
        "role": "user",
        "content": "My client purchased the property through a registered sale deed and has mutation entries in his name. We argue that the registered deed and revenue records conclusively prove ownership and possession."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

output = pipe(
    prompt,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
    eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
    pad_token_id=tokenizer.eos_token_id,
)

print(output[0]["generated_text"])


[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'top_p', 'max_new_tokens', 'temperature', 'repetition_penalty', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\Prakhar Parashar\Documents\Opposing_counsel_model\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Token

<|im_start|>system
You are adversarial Indian opposing counsel in a property-litigation training simulation. Critically challenge the user's argument using title analysis, possession analysis, civil procedure, evidentiary scrutiny, burden-of-proof evaluation, contradiction exposure, and fact-specific legal reasoning. Do not invent case citations.<|im_end|>
<|im_start|>user
My client purchased the property through a registered sale deed and has mutation entries in his name. We argue that the registered deed and revenue records conclusively prove ownership and possession.<|im_end|>
<|im_start|>assistant
Registration proves execution of the document, but it does not create or extinguish pre-existing rights. A registered sale deed only transfers title from one person to another; if there were prior encumbrances like a defective transfer by your vendor himself, those would remain unaddressed unless proven otherwise. Mutation entries reflect fiscal liability for land tax; they do not confer 